In [1]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q kaggle mlflow wandb

    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    !mlflow db upgrade sqlite:///mlflow.db

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

Upload your kaggle.json


Saving kaggle.json to kaggle.json
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adola23 (adola23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100% 2.70M/2.70M [00:00<00:00, 128MB/s]

2026/07/11 04:23:26 INFO mlflow.store.db.utils: Updating database tables


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.base import BaseEstimator
from sklearn.pipeline import Pipeline
import mlflow
import mlflow.sklearn
import wandb

In [3]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
train.shape

(421570, 5)

In [4]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [5]:
class MovingAvg(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        left = (self.kernel_size - 1) // 2
        right = self.kernel_size - 1 - left
        front = x[:, 0:1].repeat(1, left)
        end = x[:, -1:].repeat(1, right)
        x = torch.cat([front, x, end], dim=1)
        return self.avg(x.unsqueeze(1)).squeeze(1)

class DLinear(nn.Module):
    def __init__(self, seq_len, pred_len=1, kernel_size=25):
        super().__init__()
        self.decomp = MovingAvg(kernel_size)
        self.linear_trend = nn.Linear(seq_len, pred_len)
        self.linear_seasonal = nn.Linear(seq_len, pred_len)

    def forward(self, x):
        trend = self.decomp(x)
        seasonal = x - trend
        return self.linear_trend(trend) + self.linear_seasonal(seasonal)

class WindowDataset(Dataset):
    def __init__(self, windows, targets, weights):
        self.windows = windows
        self.targets = targets
        self.weights = weights

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx], self.targets[idx], self.weights[idx]

def build_series_matrix(df):
    pivot = df.pivot_table(index=['Store', 'Dept'], columns='Date', values='Weekly_Sales')
    pivot = pivot.sort_index(axis=1)
    holiday = df.drop_duplicates('Date').set_index('Date')['IsHoliday'].sort_index()
    holiday = holiday.reindex(pivot.columns).fillna(False)
    return pivot, holiday

def make_windows(pivot, holiday, seq_len=52):
    values = pivot.fillna(0).values
    n_series, n_time = values.shape

    windows, targets, weights = [], [], []
    for t in range(seq_len, n_time):
        window = values[:, t - seq_len:t]
        target = values[:, t]
        valid = ~pivot.iloc[:, t].isna().values
        if valid.sum() == 0:
            continue
        w = np.where(holiday.iloc[t], 5.0, 1.0)
        windows.append(window[valid])
        targets.append(target[valid])
        weights.append(np.full(valid.sum(), w))
    return np.concatenate(windows), np.concatenate(targets), np.concatenate(weights)

class DLinearForecaster(BaseEstimator):
    def __init__(self, seq_len=52, kernel_size=25, epochs=15, lr=1e-3, batch_size=512):
        self.seq_len = seq_len
        self.kernel_size = kernel_size
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size

    def fit(self, X, y, log_fn=None):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df['Weekly_Sales'] = y.values if hasattr(y, 'values') else y

        pivot, holiday = build_series_matrix(df)
        self.seq_len_ = min(self.seq_len, pivot.shape[1] - 1)
        self.series_mean_ = pivot.mean(axis=1).fillna(0)
        self.series_std_ = pivot.std(axis=1).fillna(1).replace(0, 1)

        norm_pivot = pivot.sub(self.series_mean_, axis=0).div(self.series_std_, axis=0)
        windows, targets, weights = make_windows(norm_pivot, holiday, self.seq_len_)

        X_t = torch.tensor(windows, dtype=torch.float32)
        y_t = torch.tensor(targets, dtype=torch.float32).unsqueeze(1)
        w_t = torch.tensor(weights, dtype=torch.float32).unsqueeze(1)

        loader = DataLoader(WindowDataset(X_t, y_t, w_t), batch_size=self.batch_size, shuffle=True)

        self.model_ = DLinear(self.seq_len_, 1, self.kernel_size)
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr)

        for epoch in range(self.epochs):
            epoch_loss = 0.0
            for xb, yb, wb in loader:
                opt.zero_grad()
                pred = self.model_(xb)
                loss = (wb * (pred - yb).abs()).mean()
                loss.backward()
                opt.step()
                epoch_loss += loss.item() * len(xb)
            epoch_loss /= len(loader.dataset)
            if log_fn is not None:
                log_fn(epoch, epoch_loss)

        self.history_ = pivot
        return self

    def _forecast_series(self, key, target_dates):
        if key not in self.history_.index:
            return pd.Series(self.series_mean_.mean(), index=target_dates)

        mean = self.series_mean_.loc[key]
        std = self.series_std_.loc[key]
        series = self.history_.loc[key].copy()

        max_date = max(target_dates.max(), series.index.max())
        all_dates = pd.date_range(series.index.min(), max_date, freq='W-FRI')

        series = series.reindex(all_dates)
        known_mask = series.notna()
        series = series.fillna(0)

        values = ((series - mean) / std).values.tolist()

        self.model_.eval()
        with torch.no_grad():
            for i in range(len(values)):
                if i >= self.seq_len_ and not known_mask.iloc[i]:
                    window = torch.tensor([values[i - self.seq_len_:i]], dtype=torch.float32)
                    values[i] = self.model_(window).item()

        result = pd.Series(values, index=all_dates) * std + mean
        return result.reindex(target_dates)

    def predict(self, X):
        df = X.copy().reset_index(drop=True)
        df['Date'] = pd.to_datetime(df['Date'])
        preds = np.zeros(len(df))
        for (store, dept), group in df.groupby(['Store', 'Dept']):
            forecast = self._forecast_series((store, dept), pd.DatetimeIndex(group['Date'].unique()))
            preds[group.index] = forecast.reindex(group['Date']).values
        return preds

In [6]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('DLinear_v2_Training')

2026/07/11 04:23:42 INFO mlflow.tracking.fluent: Experiment with name 'DLinear_v2_Training' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/walmart-sales-forecasting/mlruns/5', creation_time=1783743822803, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1783743822803, lifecycle_stage='active', name='DLinear_v2_Training', tags={}, trace_location=None, workspace='default'>

In [7]:
with mlflow.start_run(run_name='DLinear_v2_Cleaning'):
    wandb.init(project='walmart-sales-forecasting', name='DLinear_v2_Cleaning', reinit='finish_previous')

    n_negative = int((train['Weekly_Sales'] < 0).sum())
    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')
    wandb.log({'negative_sales_rows': n_negative})
    wandb.finish()

    y_train = train['Weekly_Sales'].clip(lower=0)

negative_sales_rows,▁
negative_sales_rows,1285


In [8]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

[(np.datetime64('2010-10-01T00:00:00.000000000'),
  np.datetime64('2011-06-03T00:00:00.000000000')),
 (np.datetime64('2011-06-03T00:00:00.000000000'),
  np.datetime64('2012-02-03T00:00:00.000000000')),
 (np.datetime64('2012-02-03T00:00:00.000000000'),
  np.datetime64('2012-10-05T00:00:00.000000000'))]

In [9]:
SEARCH_SPACE = {
    'seq_len': [26, 39, 52],
    'kernel_size': [13, 25],
    'epochs': [10, 15, 25],
    'lr': [1e-3, 3e-3],
    'batch_size': [256, 512],
}
N_TRIALS = 8

rng = np.random.default_rng(42)

def sample_params(rng):
    return {k: v[rng.integers(len(v))] for k, v in SEARCH_SPACE.items()}

def cv_score(params):
    scores = []
    for train_end, val_end in splits:
        tm = train['Date'] <= train_end
        vm = (train['Date'] > train_end) & (train['Date'] <= val_end)
        model = DLinearForecaster(**params)
        model.fit(train.loc[tm, ['Store', 'Dept', 'Date', 'IsHoliday']], y_train[tm])
        preds = model.predict(train.loc[vm, ['Store', 'Dept', 'Date', 'IsHoliday']])
        scores.append(wmae(y_train[vm].values, preds, train.loc[vm, 'IsHoliday'].values))
    return scores

with mlflow.start_run(run_name='DLinear_v2_Tuning'):
    wandb.init(project='walmart-sales-forecasting', name='DLinear_v2_Tuning', reinit='finish_previous')

    best_wmae, best_params = None, None
    for t in range(N_TRIALS):
        params = sample_params(rng)
        scores = cv_score(params)
        mean = float(np.mean(scores))
        mlflow.log_metric('trial_wmae_mean', mean, step=t)
        wandb.log({'trial': t, 'trial_wmae_mean': mean, **params})
        if best_wmae is None or mean < best_wmae:
            best_wmae, best_params = mean, params
        print(t, round(mean, 1), params)

    mlflow.log_metric('best_wmae_mean', best_wmae)
    mlflow.log_dict(best_params, 'best_params.json')
    wandb.log({'best_wmae_mean': best_wmae})
    wandb.finish()

best_wmae, best_params

0 3316.2 {'seq_len': 26, 'kernel_size': 25, 'epochs': 15, 'lr': 0.001, 'batch_size': 256}
1 3135.6 {'seq_len': 52, 'kernel_size': 13, 'epochs': 25, 'lr': 0.001, 'batch_size': 256}
2 3452.2 {'seq_len': 39, 'kernel_size': 25, 'epochs': 25, 'lr': 0.003, 'batch_size': 512}
3 3102.6 {'seq_len': 52, 'kernel_size': 25, 'epochs': 10, 'lr': 0.003, 'batch_size': 256}
4 3399.8 {'seq_len': 39, 'kernel_size': 13, 'epochs': 10, 'lr': 0.003, 'batch_size': 512}
5 3446.3 {'seq_len': 39, 'kernel_size': 13, 'epochs': 25, 'lr': 0.003, 'batch_size': 256}
6 3354.6 {'seq_len': 39, 'kernel_size': 13, 'epochs': 10, 'lr': 0.003, 'batch_size': 512}
7 3313.8 {'seq_len': 26, 'kernel_size': 25, 'epochs': 25, 'lr': 0.001, 'batch_size': 512}


batch_size,▁▁█▁█▁██
best_wmae_mean,▁
epochs,▃██▁▁█▁█
kernel_size,█▁██▁▁▁█
lr,▁▁█████▁
seq_len,▁█▅█▅▅▅▁
trial,▁▂▃▄▅▆▇█
trial_wmae_mean,▅▂█▁▇█▆▅
batch_size,512
best_wmae_mean,3102.61994
epochs,25


(3102.6199383445087,
 {'seq_len': 52,
  'kernel_size': 25,
  'epochs': 10,
  'lr': 0.003,
  'batch_size': 256})

In [10]:
with mlflow.start_run(run_name='DLinear_v2_CV'):
    mlflow.log_params(best_params)
    wandb.init(project='walmart-sales-forecasting', name='DLinear_v2_CV', config=best_params, reinit='finish_previous')

    fold_scores = cv_score(best_params)
    for i, s in enumerate(fold_scores):
        mlflow.log_metric('wmae', s, step=i)
        wandb.log({'fold': i, 'wmae': s})

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))
    wandb.log({'wmae_mean': float(np.mean(fold_scores)), 'wmae_std': float(np.std(fold_scores))})
    wandb.finish()

fold_scores

fold,▁▅█
wmae,█▅▁
wmae_mean,▁
wmae_std,▁
fold,2
wmae,2015.15423
wmae_mean,3170.61857
wmae_std,924.68635


[np.float64(4278.675077155393),
 np.float64(3218.0263849642743),
 np.float64(2015.15423473458)]

In [11]:
with mlflow.start_run(run_name='DLinear_v2_Final'):
    mlflow.log_params(best_params)
    wandb.init(project='walmart-sales-forecasting', name='DLinear_v2_Final', config=best_params, reinit='finish_previous')

    pipeline = Pipeline([
        ('model', DLinearForecaster(**best_params)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train)

    preds = pipeline.predict(train[['Store', 'Dept', 'Date', 'IsHoliday']])
    train_wmae = wmae(y_train.values, preds, train['IsHoliday'].values)
    mlflow.log_metric('train_wmae', train_wmae)
    wandb.log({'train_wmae': train_wmae})

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')
    wandb.finish()

train_wmae

2026/07/11 04:54:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/11 04:54:45 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


train_wmae,▁
train_wmae,0.0


np.float64(2.8204687984997125e-14)

In [12]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Run model_experiment_DLinear_v2.ipynb in Colab"
    !git push

[main 61d86c3] Run model_experiment_DLinear_v2.ipynb in Colab
 7 files changed, 68 insertions(+)
 create mode 100644 mlruns/5/6a48a964548f43658a490d7f71502a28/artifacts/best_params.json
 create mode 100644 mlruns/5/models/m-d1484bf886cf4b0bbd8ba2fa37df7edf/artifacts/MLmodel
 create mode 100644 mlruns/5/models/m-d1484bf886cf4b0bbd8ba2fa37df7edf/artifacts/conda.yaml
 create mode 100644 mlruns/5/models/m-d1484bf886cf4b0bbd8ba2fa37df7edf/artifacts/model.pkl
 create mode 100644 mlruns/5/models/m-d1484bf886cf4b0bbd8ba2fa37df7edf/artifacts/python_env.yaml
 create mode 100644 mlruns/5/models/m-d1484bf886cf4b0bbd8ba2fa37df7edf/artifacts/requirements.txt
Enumerating objects: 19, done.
Counting objects: 100% (19/19), done.
Delta compression using up to 2 threads
Compressing objects: 100% (13/13), done.
Writing objects: 100% (16/16), 1.78 MiB | 2.31 MiB/s, done.
Total 16 (delta 3), reused 3 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://gi